<a href="https://colab.research.google.com/github/Autosnight/COMP90042_2026/blob/andrew-branch/embedding_to_classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#installing libraries
%pip install transformers

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression #using log reg due to quick training and inference, good baseline measure
import torch
import transformers as ppb
import warnings
warnings.filterwarnings('ignore')

# Loading data
/content/evidence.json
and /content/COMP90042_2026/data/train-claims.json
convert claims into table with 2 colums: claim text and labels
convert evidence into list of strings

In [ ]:
import json
import pandas as pd

# Load claims data
with open('/content/COMP90042_2026/data/train-claims.json', 'r') as f:
    claims_data = json.load(f)

# Convert claims to DataFrame (claim text and labels)
claims_df = pd.DataFrame.from_dict(claims_data, orient='index')
claims_df = claims_df[['claim_text', 'label']]

# Load evidence data
with open('/content/evidence.json', 'r') as f:
    evidence_data = json.load(f)

# Convert evidence into a list of strings
evidence_list = list(evidence_data.values())

display(claims_df.head())
print(f"Loaded {len(evidence_list)} evidence items.")

# Loading frozen models + embeddings
1 cell: load distilbert and jina v5. jina v5 retrieval nano model.
different cell: use sentence transformer to convert all claims and evidence into jina embeddings (jina.encode). claims should be query embeddings, evidence should be document embeddings


In [ ]:
%pip install -U sentence-transformers
from sentence_transformers import SentenceTransformer

# Load Jina v5 retrieval nano model
jina_model = SentenceTransformer('jinaai/jina-embeddings-v3', trust_remote_code=True)

# Note: DistilBert is already loaded as 'frozen_bert' in a previous cell.

In [ ]:
# Generate embeddings
# Claims as queries, Evidence as documents
claim_texts = claims_df['claim_text'].tolist()

print("Encoding claims...")
claim_embeddings = jina_model.encode(claim_texts, task="retrieval.query")

print("Encoding evidence (this may take a moment)...")
evidence_embeddings = jina_model.encode(evidence_list, task="retrieval.passage")

In [ ]:
#retrieve class and weights
model_class, pretrained_weights = (ppb.DistilBertModel, 'distilbert-base-uncased')
#load model
frozen_bert = model_class.from_pretrained(pretrained_weights)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


# retrieval
use jina similarity function with jina embeddings to get 2 most similar texts from documents/evidence for each claim. add them to a dictionary with key = claim (string) and value = evidences (string)

In [ ]:
import numpy as np

claim_evidence_map = {}

# Calculate similarity using jina_model's similarity function
# jina_model.similarity returns a matrix of cosine similarities
similarities = jina_model.similarity(claim_embeddings, evidence_embeddings)

# Convert to numpy if it's a tensor for easier indexing
if hasattr(similarities, 'numpy'):
    similarities = similarities.numpy()

for i, claim in enumerate(claim_texts):
    # Get indices of top 2 highest similarity scores
    top_2_indices = np.argsort(similarities[i])[-2:][::-1]
    retrieved_evidence = [evidence_list[idx] for idx in top_2_indices]
    claim_evidence_map[claim] = retrieved_evidence

print(f"Mapped {len(claim_evidence_map)} claims to their top 2 evidence pieces using Jina similarity.")

# concatination:
concatinate these strings together and add sep tokens between and then encode them for distilbert with padding  

In [ ]:
tokenizer = ppb.DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

concatenated_inputs = []
for claim, evidences in claim_evidence_map.items():
    # Concatenate: [CLS] Claim [SEP] Evidence1 [SEP] Evidence2 [SEP]
    text = f"{claim} [SEP] {evidences[0]} [SEP] {evidences[1]}"
    concatenated_inputs.append(text)

tokenized_features = tokenizer(
    concatenated_inputs,
    padding='max_length',
    truncation=True,
    max_length=512,
    return_tensors='pt'
)

# classifier embeddings
use distilbert to create cls embeddings of concatinated claim-evidence-evidence strings

In [ ]:
frozen_bert.eval()
all_features = []

with torch.no_grad():
    # Processing in small batches to avoid OOM
    batch_size = 8
    for i in range(0, len(concatenated_inputs), batch_size):
        batch_ids = tokenized_features['input_ids'][i:i+batch_size]
        batch_mask = tokenized_features['attention_mask'][i:i+batch_size]

        outputs = frozen_bert(batch_ids, attention_mask=batch_mask)
        # Extract [CLS] token embeddings (first token)
        cls_embeddings = outputs.last_hidden_state[:, 0, :].numpy()
        all_features.append(cls_embeddings)

features = np.concatenate(all_features, axis=0)
labels = claims_df['label'].values
print(f"Feature matrix shape: {features.shape}")

#

# Training classification model

---
## input:

features and labels,

Output: accuracy scores in train and test set


In [ ]:
# Note: 'labels' must be defined before running this cell
# labels = np.array(...)

# Splitting the data
train_features, test_features, train_labels, test_labels = train_test_split(features, labels)

# Initialize and train the Logistic Regression model
completed_classifier = LogisticRegression()
completed_classifier.fit(train_features, train_labels)